# Step 2: Screenshot to Action Agent Loop

**Topic:** Closing the loop between a screenshot, a vision model, and a PyAutoGUI action

**Resources used:**
- PyAutoGUI docs (pyautogui.readthedocs.io) and Step 1 (`01-pyautogui-basics/pyautogui_basics.py`)
- mss docs (python-mss.readthedocs.io)
- Groq vision docs (console.groq.com/docs/vision), checked on 2026-08-31. Current production vision models are `qwen/qwen3.6-27b` and `qwen/qwen3.8-27b`. This notebook uses `qwen/qwen3.8-27b` since it is the newer of the two with tunable reasoning effort. If it gets deprecated later, swap the `VISION_MODEL` constant in Section 2, nothing else needs to change.

**Summary:** Captures the screen with mss, sends it to a Groq vision model for a next action, parses that action, executes it with PyAutoGUI, and repeats with a hard step limit, ending in a calculator mini-project.


## Setup

Run this cell first, everything below depends on it.

In [2]:
!pip install pyautogui mss groq


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
import time
import json
import base64

from dotenv import load_dotenv

import pyautogui
import mss
import mss.tools
from groq import Groq

load_dotenv()  # reads .env in the current working directory into os.environ

# GROQ_API_KEY must be set as an environment variable, never hardcode a key in a notebook
# that might get committed to GitHub
client = Groq(api_key=os.environ["GROQ_API_KEY"])

## Section 1: Screenshot capture and base64 encoding

Reuses the mss pattern from Step 1. The model can't read a local file path, so the PNG
bytes get base64 encoded and sent as a `data:image/png;base64,...` URI instead of a
public `image_url`.

In [5]:
def capture_screenshot_base64(save_path="step_screenshot.png"):
    # monitors[1] is the primary display, monitors[0] would be all monitors stitched
    # together, which is usually not what a single-screen agent loop wants
    with mss.mss() as sct:
        monitor = sct.monitors[1]
        shot = sct.grab(monitor)
        png_bytes = mss.tools.to_png(shot.rgb, shot.size)

    # keep the file on disk too, useful for debugging what the model actually saw
    with open(save_path, "wb") as f:
        f.write(png_bytes)

    b64_string = base64.b64encode(png_bytes).decode("utf-8")
    return b64_string, save_path


Test this section on its own before moving on.

In [6]:
b64_image, saved_path = capture_screenshot_base64()
print(f"Saved screenshot to {saved_path}, base64 length: {len(b64_image)} chars")


Saved screenshot to step_screenshot.png, base64 length: 363176 chars


C:\Users\zaras\AppData\Local\Temp\ipykernel_15740\529249015.py:4: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


## Section 2: Ask the vision model for a next action

The model is instructed to answer only in JSON, this is what section 3 parses. Screen
size gets passed in the prompt because the model has no other way to know the resolution
it should return coordinates in.

In [7]:
VISION_MODEL = "qwen/qwen3.8-27b"  # current production vision model on Groq, checked 2026-08-31

def ask_model_for_next_action(base64_image, task_description, history=None):
    screen_width, screen_height = pyautogui.size()
    history = history or []

    system_instructions = (
        f"You control a computer by looking at screenshots. The screen is "
        f"{screen_width}x{screen_height} pixels, origin (0,0) is the top-left corner. "
        f"Task: {task_description}\n"
        f"Steps taken so far: {history}\n"
        "Respond with ONLY a JSON object, no other text, in this exact shape: "
        '{"action": "click"|"double_click"|"type"|"done", '
        '"x": <int or null>, "y": <int or null>, "text": <string or null>, '
        '"reasoning": "<short reason>"}. '
        "Use action \"done\" once the task is visibly complete."
    )

    response = client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": system_instructions},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                    },
                ],
            }
        ],
        # json_object mode forces valid JSON back, which section 3 relies on instead
        # of trying to regex a next action out of free-form text
        response_format={"type": "json_object"},
        temperature=0.2,
        max_completion_tokens=512,
    )

    return response.choices[0].message.content


Test this section on its own using the screenshot captured above.

In [8]:
raw_response = ask_model_for_next_action(
    b64_image,
    task_description="Just describe the top-left icon on the screen, respond with action done.",
)
print(raw_response)


{"action": "done", "x": null, "y": null, "text": null, "reasoning": "Described the top-left icon (VS Code logo) as requested."}


## Section 3: Parse the model response into a next action

Even with `response_format=json_object`, wrap the parse in a try/except. A model can
still return JSON that is valid but missing keys the loop needs, so this fills sane
defaults instead of crashing the whole loop on one bad step.

In [9]:
def parse_next_action(raw_response):
    try:
        action = json.loads(raw_response)
    except json.JSONDecodeError:
        # if the model ever breaks format, treat it as done rather than looping blind
        return {"action": "done", "x": None, "y": None, "text": None, "reasoning": "unparseable response"}

    action.setdefault("action", "done")
    action.setdefault("x", None)
    action.setdefault("y", None)
    action.setdefault("text", None)
    action.setdefault("reasoning", "")
    return action


In [10]:
parsed = parse_next_action(raw_response)
print(parsed)


{'action': 'done', 'x': None, 'y': None, 'text': None, 'reasoning': 'Described the top-left icon (VS Code logo) as requested.'}


## Section 4: Execute the action with PyAutoGUI

Coordinates come from the model, not from you, so they get validated against the actual
screen size before any mouse call runs. This is the main risk point in the whole loop,
a bad coordinate here is a click somewhere you did not intend.

In [11]:
def execute_action(action):
    screen_width, screen_height = pyautogui.size()

    if action["action"] == "done":
        print("Model signaled done:", action["reasoning"])
        return True  # loop should stop

    if action["action"] in ("click", "double_click"):
        x, y = action["x"], action["y"]
        if x is None or y is None or not (0 <= x <= screen_width and 0 <= y <= screen_height):
            print("Skipping action, coordinates out of range or missing:", action)
            return False
        pyautogui.moveTo(x, y, duration=0.3)
        if action["action"] == "click":
            pyautogui.click()
        else:
            pyautogui.doubleClick()

    elif action["action"] == "type":
        if action["text"]:
            pyautogui.write(action["text"], interval=0.05)

    return False  # loop should continue


Test with a manually built action instead of a model response, so a bad model output can't move the mouse during this isolated test.

In [12]:
test_action = {"action": "done", "x": None, "y": None, "text": None, "reasoning": "manual test"}
execute_action(test_action)


Model signaled done: manual test


True

## Section 5: The full loop

`max_steps` is the hard limit mentioned in the fail-safe discussion from Step 1, without
it a model that never emits `done` would run forever. `time.sleep` between steps gives
the target app time to visually update before the next screenshot is taken.

In [13]:
def run_agent_loop(task_description, max_steps=15, delay_between_steps=1.0):
    history = []

    for step in range(1, max_steps + 1):
        b64_image, _ = capture_screenshot_base64(save_path=f"loop_step_{step}.png")
        raw_response = ask_model_for_next_action(b64_image, task_description, history)
        action = parse_next_action(raw_response)

        print(f"Step {step}: {action['action']} -> {action.get('reasoning', '')}")
        history.append(action)

        is_done = execute_action(action)
        if is_done:
            print(f"Loop finished after {step} step(s)")
            return history

        time.sleep(delay_between_steps)

    print(f"Hit max_steps ({max_steps}) without the model signaling done")
    return history


## Section 6: Mini project, calculator end to end

Opening the calculator app is done with a keyboard-only OS shortcut, not a hardcoded
screen coordinate, since coordinates for a launcher icon are not portable across
machines. The example below is Windows (`win` key opens Start search). On macOS,
replace that first cell with `pyautogui.hotkey("command", "space")` then
`pyautogui.write("calculator")` then `pyautogui.press("enter")`. On Linux this depends
on your desktop environment's application launcher shortcut.

Once the calculator window is open, the rest of the task (clicking digits and the
equals sign) is handled entirely by `run_agent_loop`, no coordinates are hardcoded
there, the model reads each screenshot and decides where to click.

In [14]:
# Open the calculator app (Windows), give it a moment to launch
pyautogui.press("win")
time.sleep(0.5)
pyautogui.write("calculator", interval=0.05)
pyautogui.press("enter")
time.sleep(1.5)  # let the app window fully render before the loop starts screenshotting


In [15]:
calculator_task = (
    "This is a calculator app. Click the on-screen buttons to compute 12 + 7, "
    "then click equals so the result is visible. Respond with action done once "
    "the result is shown on screen."
)

final_history = run_agent_loop(calculator_task, max_steps=15)


C:\Users\zaras\AppData\Local\Temp\ipykernel_15740\529249015.py:4: DeprecationWarning: mss.mss is deprecated and will be removed in a future release; use mss.MSS instead
  with mss.mss() as sct:


Step 1: click -> Click the digit 1 to begin entering 12.
Step 2: click -> Click the digit 2 to complete entering 12.
Step 3: click -> Click the plus (+) operator to add to 12.
Step 4: click -> Click the digit 7 to enter the second operand.
Step 5: click -> Click the equals button to compute 12 + 7 and show the result.
Step 6: done -> The equals button was clicked and the calculator should now display the result of 12 + 7.
Model signaled done: The equals button was clicked and the calculator should now display the result of 12 + 7.
Loop finished after 6 step(s)
